In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
data=pd.read_csv(r"C:\Users\HP\Desktop\notebook python\hotel_bookings.csv")
df=data.copy()
df

DATA SHAPE


In [ ]:
df.describe()

In [ ]:
df.describe

In [ ]:
df.shape
print(f"no of rows = {df.shape[0]}")
print(f"no of columns = {df.shape[1]}")

In [ ]:
print(f"columns name  = {df.columns.tolist()}")


In [ ]:
df.head(2)

In [ ]:
df.dtypes

In [ ]:
df['children'] = df['children'].fillna(0).astype(int)
df['agent'] = df['agent'].fillna(0).astype(int)
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])


df.head(2)

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df['country']   = df['country'].fillna('Unknown')
df['company']   = df['company'].fillna(0)

after editing the data and removing the nulls and convert it to the correct data type

In [ ]:

df[['adr', 'lead_time', 'adults', 'children', 'babies']].describe()
# as it appeared in the adr average daily rate couldnt be negative

In [ ]:
df['adr'] = df['adr'].clip(lower=0)

putting any adr less than 0 to be 0  as it couldnt be negative value

In [ ]:
# Total nights stayed
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# Total guests
df['total_guests'] = df['adults'] + df['children'] + df['babies']

# Combine date parts into one column
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + ' ' +
    df['arrival_date_month'] + ' ' +
    df['arrival_date_day_of_month'].astype(str),
    format='%Y %B %d'
)
df.head(2)

adding three columns to the data to make it readable  
as total nights to know how much night the visitor stays 
the number of the whole visitor 
the date of arrival 

In [ ]:
df[df['adults'] == 0].shape

In [ ]:
df = df[df['adults'] > 0]

removing rows with 0 adults as there is no reservation could happen with children or babies 


In [ ]:
df.shape
df.columns

In [ ]:
df['hotel'].unique()

In [ ]:
df['market_segment'].unique()

In [ ]:
df['is_repeated_guest'].unique()

In [ ]:
df['distribution_channel'].unique()

QUESTIONS


1. What is the number of cancellations for each hotel?
2. What is the  number of guests for each hotel?
3. Is the hotel service good enough for guests to come again?
4. What is the most reserved room type?
5. What is the average number of nights people stay?
6. Are weekend night reservations more than weekday night reservations?
7. Do most people come as adults only or with children?
8. What is the average lead time?
9. Which country visits the hotel the most?
10. What is the most ordered meal type?
11. Did all guests reserve the same room type that was assigned to them?
12. What is the average number of days on the waiting list?
13. What is the most common booking method?
14. What is the date with the highest number of reservations?
15. Do people prefer to pay a deposit?
16. What are the ways people know about the hotels?


In [ ]:
#what is the number of cancellation with the hotel

R_of_cancelation = df.groupby('hotel')['is_canceled'].mean() * 100

R_of_cancelation.plot(kind='bar')

plt.xlabel('Hotel Name')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate for Each Hotel', fontsize=14, fontweight='bold')

plt.show()

print(R_of_cancelation.round(2))


In [ ]:
#2. What is the  number of guests for each hotel?
total_visitor = df.groupby('hotel')['total_guests'].sum() 
total_visitor.plot(kind='bar')

plt.title('total_visitor', fontsize=12, fontweight='bold')
plt.xlabel('Hotel Type')
plt.ylabel('Number of guests')

In [ ]:
#3. Is the hotel service good enough for guests to come again?
satisfied_service = df.groupby('hotel')['is_repeated_guest'].sum() 
satisfied_service.plot(kind='bar')

plt.title('satisfied_service', fontsize=12, fontweight='bold')
plt.xlabel('Hotel Type')
plt.ylabel('Number of guests visit again')
plt.xticks(rotation=45)


In [ ]:
#4. What is the most reserved room type?
room_counts = df.groupby("hotel")['reserved_room_type'].value_counts().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='reserved_room_type',hue="hotel" ,y='count', data=room_counts, palette='Set2', ax=ax)

ax.set_title('Most Frequently Reserved Room Types', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Room Type Code', fontsize=10)
ax.set_ylabel('Number of Bookings', fontsize=10)

plt.tight_layout()

In [ ]:
#5. What is the average number of nights people stay?
yearly_avg_nights = df[df['is_canceled'] == 0].groupby(['arrival_date_year', 'hotel'])['total_nights'].sum().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))

sns.lineplot(
    x='arrival_date_year', 
    y='total_nights', 
    hue='hotel', 
    data=yearly_avg_nights, 
    marker='o',          
    markersize=8,
    linewidth=2.5,
    palette='Set2', 
    ax=ax
)
ax.set_title('Average Length of Stay (Nights) Over the Years', fontsize=12, fontweight='bold', pad=11)
ax.set_xlabel('Arrival Year', fontsize=10)
ax.set_ylabel('Average Number of Nights Stayed', fontsize=10)

ax.set_xticks(yearly_avg_nights['arrival_date_year'].unique())

plt.tight_layout()

In [ ]:

#6. Are weekend night reservations more than weekday night reservations?
total_weekend_nights = df['stays_in_weekend_nights'].sum()
total_weekday_nights = df['stays_in_week_nights'].sum()

labels = ['Weekday Nights', 'Weekend Nights']
sizes = [total_weekday_nights, total_weekend_nights]
colors = sns.color_palette('Set2')[:2]

# 3. Create the pie chart using subplots (avoiding .figure() to follow constraints)
fig, ax = plt.subplots(figsize=(6, 6))

# 4. Plot the pie chart
ax.pie(
    sizes, 
    labels=labels, 
    autopct='%1.1f%%',   # Displays the exact percentage on the slice
    startangle=140,      # Rotates the starting position for a clean look
    colors=colors,

)

# 5. Add a professional title
ax.set_title('Comparison of Total Weeknight vs. Weekend Night Reservations')

# 6. Apply tight layout and save the visualization
plt.tight_layout()


In [ ]:
df.columns

In [ ]:
#77. Do most people come as adults only or with children?
df_confirmed = df[df['is_canceled'] == 0]
adults_only = df[(df['children'] == 0) & (df['babies'] == 0)].shape[0]

with_children_or_babies = df[(df['children'] > 0) | (df['babies'] > 0)].shape[0]

values = [adults_only, with_children_or_babies]
labels = ['Adults Only', 'With Children/Babies']

plt.pie(values, labels=labels, autopct='%1.1f%%')
plt.title('Adults Only vs With Children/Babies')
plt.show()

In [ ]:

#8. What is the average lead time?lead_time
avg_lead_time = df['lead_time'].mean()
avg_lead_time



9. Which country visits the hotel the most?
10. What is the most ordered meal type?
11. Did all guests reserve the same room type that was assigned to them?
12. What is the average number of days on the waiting list?
13. What is the most common booking method?
14. What is the date with the highest number of reservations?
15. Do people prefer to pay a deposit?
16. What are the ways people know about the hotels?

In [ ]:
#9
# Top 10 countries visiting the hotel
top_countries = df['country'].value_counts().head(10)

top_countries.plot(kind='bar')

plt.xlabel('Country')
plt.ylabel('Number of Reservations')
plt.title('Top 10 Countries Visiting the Hotel', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)

plt.show()


In [ ]:
#10
# Top meals in  the hotel
top_meal_ordered = df['meal'].value_counts()

top_meal_ordered.plot(kind='bar')

plt.xlabel('meal')
plt.ylabel('Number of orders')
plt.title('Top  meals ordered in  the Hotel', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)

plt.show()


In [ ]:
df['reserved_room_type'].eq(df['assigned_room_type']).value_counts().plot(
    kind='pie',
    labels=['reserved = assigned Room', 'Different Room'],
    autopct='%1.1f%%',
    colors=['steelblue', 'salmon'],
    figsize=(6, 6),
    title='Reserved vs Assigned Room Type'
)
plt.ylabel('')
plt.show()

In [ ]:
#12. What is the average number of days on the waiting list?
avg_Waiting_time = df.groupby("hotel")["days_in_waiting_list"].mean()
avg_Waiting_time.plot(kind="bar")

plt.xlabel('hotel')
plt.ylabel('average waiting days ')
plt.title('average waiting days per each hotel', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#13. What is the most common booking method?
booking_counts = df['distribution_channel'].value_counts()
booking_counts.plot(kind="bar")

plt.xlabel('the method of booking')
plt.ylabel('count of the method ')
plt.title('Method of booking', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.show()

14. What is the date with the highest number of reservations?
15. Do people prefer to pay a deposit?
16. What are the ways people know about the hotels?

In [ ]:
#= df.groupby('arrival_date')['hotel'].count()
the_perfect_date = df[df['is_canceled'] == 0].groupby('arrival_date' )['hotel'].sum().reset_index()

the_perfect_date.plot(kind="bar")

plt.xlabel('the method of booking')
plt.ylabel('count of the method ')
plt.title('Method of booking', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.show()
